## Setup

*You must run the cells in this section each time you connect to a new runtime. For example, when you return to the notebook after an idle timeout, when the runtime crashes, or when you restart or factory reset the runtime.*

Install requirements (*Note: ocdskingfishercolab installs google-colab, which expects specific versions of pandas and numpy*):

In [ ]:
! pip install --upgrade pip > pip.log
! pip install --upgrade ocdskingfishercolab ipywidgets psycopg2-binary >> pip.log

In [ ]:
# @title Import packages and load extensions { display-mode: "form" }

import gzip
import json
import os
import shutil
import tempfile
from collections import Counter
from datetime import datetime, timezone
from pathlib import Path

import numpy as np
import pandas as pd
from dateutil.relativedelta import relativedelta
from google.colab.data_table import DataTable
from google.colab.files import download
from ipywidgets import widgets
from ocdskingfishercolab import (
    authenticate_gspread,
    calculate_coverage,
    download_dataframe_as_csv,
    format_thousands,
    render_json,
    save_dataframe_to_sheet,
    save_dataframe_to_spreadsheet,
    set_dark_mode,
    set_light_mode,
)

# Load https://pypi.org/project/ipython-sql/
%load_ext sql
# Load https://colab.research.google.com/notebooks/data_table.ipynb
%load_ext google.colab.data_table

In [ ]:
# @title Configure the notebook environment { display-mode: "form" }

# Increase max columns so that Pandas DataFrames with many columns are rendered as data tables.
DataTable.max_columns = 50
# Remove the index from data tables for easier copy-pasting to Google Docs.
DataTable.include_index = False

# Return Pandas DataFrames instead of regular result sets.
%config SqlMagic.autopandas = True
# Don't print number of rows affected.
%config SqlMagic.feedback = False

# If you set Tools > Settings > Site > Theme to dark, uncomment this line.
# set_dark_mode()
# If you are creating plots to copy-paste into reports, uncomment this line.
# set_light_mode()

## Usability analysis setup

Use this section to setup the functions needed to perform a usability analysis of the dataset, to identify if a publisher has the necessary fields to calculate 71 procurement indicators related to market opportunity (market description, competition, supplier performance), value for money, internal efficiency, public integrity and service delivery.  For an OCDS publisher, it also calculates the proportion of unique procedures for which it is possible to calculate the indicator (coverage).

The usability checks includes all the indicators listed on [OCP's use case guide](https://docs.google.com/spreadsheets/d/1j-Y0ktZiOyhZzi-2GSabBCnzx6fF5lv8h1KYwi_Q9GM/edit#gid=1183427361) and the [Indicators to diagnose the performance of a procurement market document](https://docs.google.com/document/d/1vSJk9-qWSTQEx9ZZc7BUhQZMHvTRcyDYVS2sl8HB__k/edit#heading=h.nrnq1ajwwpqe).

In [ ]:
# @title Usability functions { display-mode: "form" }

RELEVANT_RULES = {
    "who": [
        "buyer/id",
        "buyer/name",
        "tender/procuringEntity/id",
        "tender/procuringEntity/name",
    ],
    "bought what": [
        "tender/items/classification/id",
        "awards/items/classification/id",
        "contracts/items/classification/id",
        "tender/items/classification/description",
        "awards/items/classification/description",
        "contracts/items/classification/description",
        "tender/items/description",
        "awards/items/description",
        "contracts/items/description",
        "tender/description",
        "awards/description",
        "contracts/description",
        "tender/title",
        "awards/title",
        "contracts/title",
    ],
    "from whom": [
        "awards/suppliers/id",
        "awards/suppliers/name",
    ],
    "for how much": [
        "awards/value/amount",
        "contracts/value/amount",
        [
            "awards/items/quantity",
            "awards/items/unit/value/amount",
        ],
        [
            "contracts/items/quantity",
            "contracts/items/unit/value/amount",
        ],
    ],
    "when": [
        "tender/tenderPeriod/endDate",
        "awards/date",
        "contracts/dateSigned",
    ],
    "how": [
        "tender/procurementMethod",
        "tender/procurementMethodDetails",
    ],
}


def check_usability_indicators(lang, result):
    # Use case guide: Indicators linked to OCDS #public
    if lang.value == "English":
        spreadsheet_key = "1j-Y0ktZiOyhZzi-2GSabBCnzx6fF5lv8h1KYwi_Q9GM"
    else:  # [ES]
        spreadsheet_key = "1l_p_e1iNUUuR5AObTJ8EY9VrcCLTAq3dnG_Fj73UH9w"

    rows = authenticate_gspread().open_by_key(spreadsheet_key).get_worksheet(0).get_all_values()
    indicators = pd.DataFrame(rows).pipe(lambda df: df.rename(columns=df.iloc[0]).drop(df.index[0]))

    if lang.value == "English":
        return result.merge(indicators.iloc[:, [0, 3, 4, 9]], on="U_id")

    return (
        indicators.iloc[:, [0, 3, 4, 5, 9]]
        .merge(result, on="U_id")
        .drop(columns="indicator")
        .rename(
            columns={
                "fields needed": "Campos necesarios",
                "calculation": "¿Se puede calcular?",
                "missing fields": "Campos faltantes",
                "coverage": "Cobertura",
            }
        )
        .replace({"¿Se puede calcular?": {"possible to calculate": "sí", "missing fields": "campos faltantes"}})
    )


def is_relevant(field_list):
    """
    Check if the dataset has the basic fields to answer: who bought what, from whom, for how much, when, and how.

    Each rule in RELEVANT_RULES is satisfied if ANY of its options is present:
    - String options: the field must be in field_list
    - List options: all fields in the list must be in field_list
    """
    results = []
    for rule_name, options in RELEVANT_RULES.items():
        available = []
        missing = []
        possible = False

        for option in options:
            if isinstance(option, str):
                if option in field_list:
                    available.append(option)
                    possible = True
                else:
                    missing.append(option)
            else:
                all_present = True
                for opt in option:
                    if opt in field_list:
                        available.append(opt)
                    else:
                        missing.append(opt)
                        all_present = False
                if all_present:
                    possible = True

        results.append(
            {
                "rule": rule_name,
                "possible_to_calculate": "Yes" if possible else "No",
                "available_fields": available,
                "missing_fields": missing,
            }
        )

    df = pd.DataFrame(results)
    return (df["possible_to_calculate"] == "Yes").all(), df

## Setup download data from the Data Registry

In [ ]:
# @title Data registry functions{ display-mode: "form" }
import requests

DATA_REGISTRY_BASE_URL = "https://data.open-contracting.org/en/"
PUBLICATIONS_URL = f"{DATA_REGISTRY_BASE_URL}publications.json"


def get_publications():
    publications = requests.get(PUBLICATIONS_URL, timeout=10).json()
    for publication in publications:
        publication["label"] = f"{publication['country']} - {publication['title']}"
    return publications


def get_publication_select_box():
    return widgets.Dropdown(
        options=sorted([entry["label"] for entry in get_publications()]),
        description="Publication:",
        disabled=False,
    )


def format_coverage(coverage):
    if not coverage:
        return pd.DataFrame(columns=["path"])
    fields = (
        pd.DataFrame.from_dict(coverage, orient="index", columns=["count"])
        .reset_index()
        .rename(columns={"index": "path"})
    )
    # Leaves only object members
    fields_table = fields[fields.path.str.contains("[a-z]$")].copy()
    fields_table["path"] = fields_table["path"].str.replace(r"[][]|^/", "", regex=True)
    return fields_table

## Select a publication from the [Data Registry](https://data.open-contracting.org/) and its field list

In [ ]:
# @title Select the publication to download { display-mode: "form" }

publication_select_box = get_publication_select_box()
publication_select_box

In [ ]:
# @title Extract the list of available fields { display-mode: "form" }

selected_publication = next(entry for entry in get_publications() if entry["label"] == publication_select_box.value)
fields_table = format_coverage(selected_publication.get("coverage", {}))

## Relevance analysis

Use this section to assess if the publication contains the required fields to answer "who bought what from whom, for how much, when and how" for some subset of contracting processes.

Generate a list of the fields published:

In [ ]:
fields_list = fields_table.iloc[:, 0].tolist()

In [ ]:
relevant, result = is_relevant(fields_list)

### Does the publication pass the relevant criterion?

In [ ]:
relevant

### Why?

In [ ]:
result

### Manually check for other fields

If the main OCDS fields are not available to answer the relevant question, you should manually check if others might be used instead. This involves not only checking for the existence of a field but its content too. For example:
- If you cannot answer "who", you could check if they disclose "buyer" or "procuringEntity" roles as part of the parties array.
- If you cannot answer "from whom", you could check if they disclose the "supplier" role as part of the parties array.

If a quick check yields no alternative field, do not spend more time. If you cannot easily find the relevant field, neither will another user.

In [ ]:
fields_table

#### Save the table to a spreadsheet

In [ ]:
spreadsheet_name = input("Enter the name of your spreadsheet:")
save_dataframe_to_sheet(spreadsheet_name, result, "relevant table")